<a href="https://colab.research.google.com/github/Imamjamiu/Imamjamiu/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Colab_Projects/mini_LLM"

# Create the folder structure
!mkdir -p "$DRIVE_PROJECT_PATH"


In [8]:
%%writefile "$DRIVE_PROJECT_PATH/train.jsonl"
{"messages":[{"role":"user","content":"Hello"}, {"role":"assistant","content":"Hi! How can I help you today?"}]}
{"messages":[{"role":"user","content":"What is your name?"}, {"role":"assistant","content":"I don't have a name, but I'm here to assist you."}]}


Writing /content/drive/MyDrive/Colab_Projects/mini_LLM/train.jsonl


In [9]:
%%writefile "$DRIVE_PROJECT_PATH/dataset.jsonl"
{"input": "hi, good afternoon", "output": "Good afternoon! How are you today?"}
{"input": "i am hungry", "output": "Sorry about that. What would you like to eat?"}
{"input": "can i get rice", "output": "Sure! Would you prefer stew or something else with it?"}
{"input": "what is the capital of france?", "output": "The capital of France is Paris."}
{"input": "thank you for the help", "output": "You're welcome! Let me know if you need anything else."}


Writing /content/drive/MyDrive/Colab_Projects/mini_LLM/dataset.jsonl


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import torch

def proper_fine_tuning_with_fixed_data():
    print("🚀 STARTING PROPER FINE-TUNING WITH FIXED DATA...")

    # Load the FIXED dataset
    dataset = load_dataset("json", data_files=fixed_dataset_path)

    print(f"📊 Loaded {len(dataset['train'])} fixed examples")

    # Load fresh model
    MODEL_NAME = "microsoft/DialoGPT-medium"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 🎯 CORRECT tokenization function
    def tokenize_dialogue(batch):
        # Format: "User: [input] Assistant: [output]"
        combined_text = [
            f"User: {inp} Assistant: {out}{tokenizer.eos_token}"
            for inp, out in zip(batch['input'], batch['output'])
        ]

        # Tokenize
        tokenized = tokenizer(
            combined_text,
            padding="max_length",
            truncation=True,
            max_length=256
        )

        tokenized["labels"] = tokenized["input_ids"]
        return tokenized

    # Tokenize
    tokenized_dataset = dataset.map(
        tokenize_dialogue,
        batched=True,
        remove_columns=["input", "output"]
    )
    tokenized_dataset.set_format("torch")

    # Training
    training_args = TrainingArguments(
        output_dir="./proper_finetune_fixed",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        learning_rate=5e-5,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
    )

    print("🎯 Training with fixed data...")
    trainer.train()

    # Save
    save_path = "/content/drive/MyDrive/Colab_Projects/Proper_ChatGPT_Model_Fixed/"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    print(f"✅ Properly fine-tuned model saved to: {save_path}")
    return save_path

# Run proper fine-tuning with fixed data
proper_model_path = proper_fine_tuning_with_fixed_data()

🚀 STARTING PROPER FINE-TUNING WITH FIXED DATA...


Generating train split: 0 examples [00:00, ? examples/s]

📊 Loaded 143 fixed examples


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

Step,Training Loss
10,2.491300
20,0.526300
30,0.495700
40,0.380900
50,0.360100
60,0.387600
70,0.324200
80,0.285300
90,0.281000
100,0.242800


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.491300
20,0.526300
30,0.495700
40,0.380900
50,0.360100
60,0.387600
70,0.324200
80,0.285300
90,0.281000
100,0.242800


✅ Properly fine-tuned model saved to: /content/drive/MyDrive/Colab_Projects/Proper_ChatGPT_Model_Fixed/


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def final_chat():
    model_path = "/content/drive/MyDrive/Colab_Projects/Proper_ChatGPT_Model_Fixed/"

    # Load from LOCAL path
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)

    print("=== FINAL WORKING CHATBOT ===")
    print("Type 'quit' to exit.\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == 'quit':
            print("Assistant: Goodbye! 👋")
            break

        prompt = f"User: {user_input} Assistant:"
        inputs = tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=60,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response[len(prompt):].strip()
        print(f"Assistant: {response}")

final_chat()

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


=== FINAL WORKING CHATBOT ===
Type 'quit' to exit.

Assistant: Hello! How can I help you?
Assistant: I’m doing great! How are you feeling?
Assistant: I'm proud of you! Keep pushing through and staying consistent, proud of where you're going.
Assistant: You're welcome! I'm here whenever you need me.
Assistant: Today, you’re learning new skills. Start with small projects, and gradually increase in difficulty.
Assistant: Yes, small projects are great. Build small projects quickly, keep track of steps, and track progress.
Assistant: Sure! Drink water, eat fruits, and sleep well.
Assistant: Try a salad. It has protein, fat and fiber.
Assistant: You can use vegetables, fruits, bread, milk, eggs, and oil. You can even add a little protein.
Assistant: Failure is learning from your mistakes. Sometimes you just have to let it go.
Assistant: Tell me something interesting—like where you want to start, or how you feel about learning, or what projects you want to start, etc
Assistant: Good progress 